# Data Science Interview Case Study: Customer Churn

This notebook demonstrates a structured end-to-end case using synthetic customer data. It is designed to teach the workflow, not to provide a production-ready churn model.

**Interview habit:** explain the business decision and target definition before selecting a model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 1. Define the problem and generate illustrative data

In [ ]:
n = 2500
tenure_months = rng.integers(1, 61, size=n)
monthly_fee = np.clip(rng.normal(72, 22, size=n), 15, 160)
support_tickets = rng.poisson(1.4, size=n)
usage_change = rng.normal(0, 1, size=n)
contract = rng.choice(["month-to-month", "annual", "two-year"], size=n, p=[0.55, 0.30, 0.15])
region = rng.choice(["North", "South", "East", "West"], size=n)
payment_method = rng.choice(["card", "bank_transfer", "wallet"], size=n)

# Synthetic relationship used only to create a plausible learning target.
logit = (
    -1.4
    + 0.035 * (monthly_fee - 70)
    - 0.028 * tenure_months
    + 0.42 * support_tickets
    - 0.55 * usage_change
    + np.where(contract == "month-to-month", 0.9, 0.0)
    + np.where(contract == "two-year", -0.7, 0.0)
)
probability = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, probability)

df = pd.DataFrame({
    "tenure_months": tenure_months,
    "monthly_fee": monthly_fee.round(2),
    "support_tickets": support_tickets,
    "usage_change": usage_change.round(3),
    "contract": contract,
    "region": region,
    "payment_method": payment_method,
    "churn": churn
})

# Introduce a small amount of missingness in feature columns.
for col in ["monthly_fee", "usage_change", "payment_method"]:
    missing_idx = rng.choice(df.index, size=int(0.025 * n), replace=False)
    df.loc[missing_idx, col] = np.nan

print("Shape:", df.shape)
print("Churn rate:", round(df["churn"].mean(), 3))
df.head()


## 2. Explore data quality and target distribution

In [ ]:
display(df.describe(include="all").T)
display(df.isna().sum().rename("missing_count").to_frame())
display(df["churn"].value_counts(normalize=True).rename("share").to_frame())


In [ ]:
df["churn"].value_counts().sort_index().plot(kind="bar")
plt.title("Synthetic churn label counts")
plt.xlabel("Churn label")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()


### Interview interpretation

- Confirm the target definition and the snapshot date in a real project.
- Features must be available when a prediction is made.
- This dataset is generated synthetically; do not interpret these relationships as real-world evidence.
- A random split is shown for a simple demonstration. If predicting future churn, prefer a time-based evaluation split.

## 3. Split data and define preprocessing

In [ ]:
X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = ["tenure_months", "monthly_fee", "support_tickets", "usage_change"]
categorical_features = ["contract", "region", "payment_method"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Train rows:", len(X_train), "| test rows:", len(X_test))


## 4. Train a baseline and logistic regression

In [ ]:
models = {
    "dummy_prior": Pipeline([
        ("preprocess", preprocessor),
        ("model", DummyClassifier(strategy="prior"))
    ]),
    "logistic_regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ])
}

results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    scores = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
        "precision_at_0.5": precision_score(y_test, pred, zero_division=0),
        "recall_at_0.5": recall_score(y_test, pred, zero_division=0),
        "f1_at_0.5": f1_score(y_test, pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df


## 5. Inspect errors and compare thresholds

In [ ]:
chosen_model = fitted_models["logistic_regression"]
test_probabilities = chosen_model.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in [0.25, 0.35, 0.50, 0.65, 0.75]:
    pred = (test_probabilities >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "customers_flagged": int(pred.sum()),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df


In [ ]:
selected_threshold = 0.50
selected_predictions = (test_probabilities >= selected_threshold).astype(int)
print("Confusion matrix at threshold", selected_threshold)
print(confusion_matrix(y_test, selected_predictions))
print(classification_report(y_test, selected_predictions, zero_division=0))


### Threshold decision

A lower threshold usually flags more customers, often increasing recall while reducing precision. The correct threshold depends on campaign capacity, incentive cost, customer value, and whether the offer actually changes churn behavior. Model risk ranking does not prove treatment effectiveness.

## 6. Permutation importance

In [ ]:
# Permutation importance evaluates the effect of shuffling each raw input feature.
perm = permutation_importance(
    chosen_model, X_test, y_test,
    scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
importance


## 7. Turn analysis into a business recommendation

In [ ]:
best_row = results_df.iloc[0]
print(
    f"Best model by held-out ROC-AUC in this run: {best_row['model']} "
    f"(ROC-AUC={best_row['roc_auc']:.3f}, "
    f"Average Precision={best_row['average_precision']:.3f})."
)
print("Use the threshold table to choose a feasible campaign volume, then test the intervention with a randomized holdout.")


### Suggested executive summary

1. **Goal:** identify customers who may churn in the defined future window.
2. **Evidence:** compare a simple baseline with a more informative model on held-out data.
3. **Operating choice:** select a threshold using campaign capacity and expected net value.
4. **Validation:** randomize eligible customers between offer and holdout groups to measure incremental retention.
5. **Risks:** synthetic data, label definition, leakage, changing customer behavior, and intervention cost.
6. **Next steps:** validate on real time-split data, assess segment-level errors and calibration, and define monitoring/rollback criteria.

Do not report the notebook's synthetic metrics as real business results.